In [1]:
# load libraries
import os,sys
sys.path.insert(0,'../src')
sys.path.insert(0,'../libs')
import squarify # pip install squarify (algorithm for treemap)
import matplotlib.pyplot as plt
import pandas as pd
from highlight_text import fig_text
import numpy as np
import plotly.express as px
from pathlib import Path
from prompts import topic_identify_simple_pt
from llm_utils import BSAgent
from dotenv import load_dotenv
# Load environment variables from project root .env file
env_path = '../.env'
load_dotenv(dotenv_path=env_path)

/ephemeral/home/xiong/miniconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Load data

In [2]:
data_dir = Path('/ephemeral/home/xiong/data/Fund/CSR')
result_dir = data_dir / 'All_AIV_2008-2024_CSV_topic_identify'

In [6]:
def load_all_results(result_dir: Path, filter_empty_topics: bool = True) -> pd.DataFrame:
    """Load and concatenate all CSV files from the result directory.
    Args:
        result_dir: Path to directory containing result CSV files
        filter_empty_topics: If True, filter out rows where topic_labels is '[]'
        
    Returns:
        Combined DataFrame containing all results
    """
    # Get list of all CSV files (excluding hidden files)
    result_files = [f for f in result_dir.glob('*.csv') if not f.name.startswith('.')]
    # Load and concatenate all files
    all_results = []
    for result_file in result_files:
        df = pd.read_csv(result_file)
        df['source_file'] = result_file.name  # Add source filename as column
        all_results.append(df)
    
    # Combine all dataframes
    combined_df = pd.concat(all_results, ignore_index=True)
    
    # Filter out empty topic labels if requested
    if filter_empty_topics:
        combined_df = combined_df[combined_df['topic_labels'] != '[]']
    
    return combined_df


In [48]:
# Load all results into single dataframe
all_results_df = load_all_results(result_dir)
print(f"Loaded {len(all_results_df):,} total rows from {all_results_df['source_file'].nunique()} files")
# Convert string representation of lists to actual lists
all_results_df['topic_labels'] = all_results_df['topic_labels'].apply(eval)


Loaded 290,370 total rows from 2038 files


In [49]:
all_results_df.head()

,File_Name,style,section,sub_section,bold_text,italic_text,par,original_text,topic_labels,confidence_scores,reasoning,source_file
0,196_2022_0,Body Text,KEY ISSUES,KEY ISSUES,Context,NaN,Context. New Zealand’s management of the pande...,Context. New Zealand’s management of the pande...,[Economic Outlook],[90.0],The paragraph discusses the economic recovery ...,results_196_2022_0.csv
1,196_2022_0,Body Text,KEY ISSUES,KEY ISSUES,Outlook and risks,NaN,Outlook and risks. Economic growth is expected...,Outlook and risks. Economic growth is expected...,[Economic Outlook],[90.0],The paragraph discusses the expected economic ...,results_196_2022_0.csv
2,196_2022_0,Body Text,KEY ISSUES,KEY ISSUES,NaN,NaN,Given the unpredictable trajectory of the pand...,Given the unpredictable trajectory of the pand...,[Economic Outlook],[90.0],The paragraph mentions the 'unpredictable traj...,results_196_2022_0.csv
3,196_2022_0,List Paragraph,KEY ISSUES,KEY ISSUES,Macroeconomic policies,NaN,Macroeconomic policies. The normalization of m...,Macroeconomic policies. The normalization of m...,[Monetary Policy],[90.0],The paragraph discusses the normalization of m...,results_196_2022_0.csv
4,196_2022_0,List Paragraph,KEY ISSUES,KEY ISSUES,Financial sector policies,NaN,Financial sector policies. Macroprudential mea...,Financial sector policies. Macroprudential mea...,[Financial Stability],[90.0],The paragraph discusses the implementation of ...,results_196_2022_0.csv


### harmonize topic labels

In [63]:
def get_topic_frequencies(df: pd.DataFrame,column_name: str) -> dict:
    """Get frequencies of all unique topic labels in the dataframe.
    
    Args:
        df: DataFrame containing 'topic_labels' column with lists of topics
        
    Returns:
        Dictionary mapping topic labels to their frequencies
    """
    # Get all unique topics and their frequencies by flattening the lists
    topic_frequencies = {}
    for topics in df[column_name]:
        if isinstance(topics, list):
            for topic in topics:
                topic_frequencies[topic] = topic_frequencies.get(topic, 0) + 1
        else:
            topic_frequencies[topics] = topic_frequencies.get(topics, 0) + 1
            
    return dict(sorted(topic_frequencies.items(), key=lambda x: x[1], reverse=True))
# Get topic frequencies
topic_freqs = get_topic_frequencies(all_results_df,'topic_labels')

In [51]:
key_phrases = '||'.join(topic_freqs.keys())
topic_definitions = topic_identify_simple_pt['user'].split('----------------')[2]
sys_message = '''
You are an experienced IMF economist. You are given:
1. A list of unique key phrases separated by “||”.
2. A set of topic definitions.

Your task is to:
- Assign each key phrase to the most appropriate topic based on the provided definitions.  
- Return the result as a dictionary (e.g., JSON) where each key is the exact key phrase (case-sensitive), and each value is the corresponding topic label (without any `**` characters).

Your output should include only this dictionary, with no additional commentary.
'''
user_message = '''
Here are the key phrases:
```
{TOPIC_KEY_PHRASES}
```

Here are the topic definitions:
```
{TOPIC_DEFINITIONS}
```

**Task:** Assign each key phrase to its most appropriate topic from the definitions.
**Format:** Return your answer in **clean JSON** with each key phrase (case-sensitive) as the key and the corresponding topic as the value. For example:

```json
{{
  "key1": "topic1",
  "key2": "topic2"
}}
```
No additional commentary or formatting is required.
'''.format(TOPIC_KEY_PHRASES=key_phrases, TOPIC_DEFINITIONS=topic_definitions)

In [52]:
# print(sys_message)
# print(user_message)


In [53]:
agent = BSAgent(
    model='gpt-4o',
    api_key=os.getenv('OPENAI_API_KEY'),
    temperature=0.1
)

In [54]:
res = agent.get_response_content(prompt_template={
                                'system': sys_message,
                                'user': user_message
                            })

In [57]:
topic_map = agent.parse_load_json_str(res)
# Verify all keys in topic_freqs exist in topic_map
missing_keys = set(topic_freqs.keys()) - set(topic_map.keys())
if missing_keys:
    print(f"Missing topic mappings for: {missing_keys}")


Missing topic mappings for: {'Debt and Fiscal Sustainability', 'Regulatory Framework is not a predefined topic, but it is a subtopic of Structural Reforms, so I will not use it as a topic label. However, I will use the closest predefined topic which is Structural Reforms.', 'Program risks'}


In [58]:
### manual update map harmonize topic labels
add_dict = {'Debt and Fiscal Sustainability': 'Fiscal Policy',
'Regulatory Framework is not a predefined topic, but it is a subtopic of Structural Reforms, so I will not use it as a topic label. However, I will use the closest predefined topic which is Structural Reforms.': 'Structural Reforms',
'Program risks': 'Other Topics'}
topic_map.update(add_dict)

In [60]:
## check again after manual update 
missing_keys = set(topic_freqs.keys()) - set(topic_map.keys())
if missing_keys:
    print(f"Missing topic mappings for: {missing_keys}")

In [61]:
# Extract first topic from topic_labels list and map to consolidated topics
all_results_df['consolidated_topic_labels'] = all_results_df['topic_labels'].apply(lambda x: topic_map.get(x[0]) if isinstance(x, list) and len(x) > 0 else None)
all_results_df.head()

,File_Name,style,section,sub_section,bold_text,italic_text,par,original_text,topic_labels,confidence_scores,reasoning,source_file,consolidated_topic_labels
0,196_2022_0,Body Text,KEY ISSUES,KEY ISSUES,Context,NaN,Context. New Zealand’s management of the pande...,Context. New Zealand’s management of the pande...,[Economic Outlook],[90.0],The paragraph discusses the economic recovery ...,results_196_2022_0.csv,Economic Outlook
1,196_2022_0,Body Text,KEY ISSUES,KEY ISSUES,Outlook and risks,NaN,Outlook and risks. Economic growth is expected...,Outlook and risks. Economic growth is expected...,[Economic Outlook],[90.0],The paragraph discusses the expected economic ...,results_196_2022_0.csv,Economic Outlook
2,196_2022_0,Body Text,KEY ISSUES,KEY ISSUES,NaN,NaN,Given the unpredictable trajectory of the pand...,Given the unpredictable trajectory of the pand...,[Economic Outlook],[90.0],The paragraph mentions the 'unpredictable traj...,results_196_2022_0.csv,Economic Outlook
3,196_2022_0,List Paragraph,KEY ISSUES,KEY ISSUES,Macroeconomic policies,NaN,Macroeconomic policies. The normalization of m...,Macroeconomic policies. The normalization of m...,[Monetary Policy],[90.0],The paragraph discusses the normalization of m...,results_196_2022_0.csv,Monetary Policy
4,196_2022_0,List Paragraph,KEY ISSUES,KEY ISSUES,Financial sector policies,NaN,Financial sector policies. Macroprudential mea...,Financial sector policies. Macroprudential mea...,[Financial Stability],[90.0],The paragraph discusses the implementation of ...,results_196_2022_0.csv,Financial Stability


### Check unique topic labels after harmonization

In [64]:
topic_freqs = get_topic_frequencies(all_results_df,'consolidated_topic_labels')
topic_freqs

{'Fiscal Policy': 78099,
 'Economic Outlook': 43181,
 'Financial Stability': 40691,
 'External Stance': 34502,
 'Other Topics': 21870,
 'Structural Reforms': 19625,
 'Monetary Policy': 16186,
 'Inclusion': 12628,
 'Governance': 12518,
 'Data Adequacy': 5827,
 'Climate Change': 4022,
 'Technology': 1221}

In [70]:
all_results_df.to_csv(data_dir / 'All_AIV_2008-2024_CSV_topic_identify_consolidated.csv',index=False)

## Visualize topic distribution

In [3]:
all_results_df=pd.read_csv(data_dir / 'All_AIV_2008-2024_CSV_topic_identify_consolidated.csv')

In [4]:
all_results_df['Year'] = all_results_df['File_Name'].str.extract(r'(\d{4})').astype(int)

###  Overall treemap

In [5]:
collapsed_df = all_results_df.groupby(['consolidated_topic_labels'], as_index=False)['File_Name'].count()
collapsed_df.head(2)

,consolidated_topic_labels,File_Name
0,Climate Change,4022
1,Data Adequacy,5827


In [6]:
def treemap_by_topic(collapsed_df,title="Treemap of AIV Topic Distribution - LLM"):
    # Create the treemap
    fig = px.treemap(collapsed_df, 
                    path=['consolidated_topic_labels'], 
                    values='File_Name',
                    #color='Name',
                    #color_continuous_scale='RdBu',
                    width=1000, 
                    height=800,
                    title=title)
    fig.update_traces(hovertemplate='labels=%{label}<br>total_count=%{value}<extra></extra>')
    return fig

In [1]:
# fig = treemap_by_topic(collapsed_df,title="Treemap of AIV Topic Distribution")
# fig.write_image(data_dir / 'figs' / "Treemap_AIV_Topic_Distribution_LLM.png")
# fig.write_html(data_dir / 'figs' / "Treemap_AIV_Topic_Distribution_LLM.html")
# fig.show()

#### Overall distribution by overtime

In [8]:
collapsed_df = all_results_df.groupby(['Year','consolidated_topic_labels'], as_index=False)['File_Name'].count()
collapsed_df.head(2)

,Year,consolidated_topic_labels,File_Name
0,2008,Climate Change,21
1,2008,Data Adequacy,328


In [9]:
def time_series_by_topic(collapsed_df,title='Topic Over Time - LLM'):
    fig = px.area(collapsed_df, 
                x="Year", 
                y="File_Name", 
                color="consolidated_topic_labels", 
                line_group="consolidated_topic_labels",
                height=500,
                width=1000,
                title=title,
                )
    fig.update_layout(legend=dict(
                                title=dict(
                                    text='High Level Topics',
                                    font=dict(size=14)  # Optional: set the font size
                                ),),
                        yaxis_title='# of paragraphs'
        
                        )
    return fig

In [2]:
# fig = time_series_by_topic(collapsed_df)
# fig.write_image(data_dir / 'figs' / "TimeSeries_AIV_Topic_counts_LLM.png")
# fig.write_html(data_dir / 'figs' / "TimeSeries_AIV_Topic_counts_LLM.html")
# fig.show()

#### Topic distribution by year by percentage

In [11]:
collapsed_df = all_results_df.groupby(['Year','consolidated_topic_labels'], as_index=False)['File_Name'].count()
collapsed_df['File_Name'] = collapsed_df.groupby('Year')['File_Name'].transform(lambda x: (x / x.sum()*100))
collapsed_df.head(2)

,Year,consolidated_topic_labels,File_Name
0,2008,Climate Change,0.154628
1,2008,Data Adequacy,2.415139


In [12]:
def time_series_level_1_percentage(collapsed_df, title = 'Topic Distribution Over Time - LLM'):
    fig = px.area(collapsed_df, 
                x="Year", 
                y="File_Name", 
                color="consolidated_topic_labels", 
                line_group="consolidated_topic_labels",
                height=500,
                width=1000,
                title=title,
                )
    fig.update_layout(legend=dict(
                                title=dict(
                                    text='High Level Topics',
                                    font=dict(size=14)  # Optional: set the font size
                                ),),
                            yaxis=dict(
                                        title='Percentage',  # Optional: Set the y-axis title
                                        range=[0, 100]  # Set the y-axis range from 0 to 100
                                    ),
        
                        )
    return fig


In [3]:
# fig = time_series_level_1_percentage(collapsed_df)
# fig.write_image(data_dir / 'figs' / "TimeSeries_AIV_Topic_Distribution_LLM.png")
# fig.write_html(data_dir / 'figs' / "TimeSeries_AIV_Topic_Distribution_LLM.html")
# fig.show()